In [18]:
import sys
import torch
import ultralytics
import cv2

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("OpenCV:", cv2.__version__)
print("GPU disponible:", torch.cuda.is_available())

AttributeError: partially initialized module 'torch' has no attribute 'types' (most likely due to a circular import)

In [2]:
!pip install roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 102.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 95.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 MB 106.5 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: numpym━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/15 [shellingham]
    Found existing installation: numpy 2.5.3━━━━━━━━━━━━━━━━━━  3/15 [shellingham]
    Uninstalling numpy-2.5.3:0m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/15 [shellingham]
      Successfully uninstalled numpy-2.5.3━━━━━━━━━━━━━━━━━━━━  3/15 [shellingham]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [roboflow]/15 [roboflow]it-py]eadless]


Conexión con roboflow y la DB

In [19]:
from roboflow import Roboflow
rf = Roboflow(api_key="G9twKyBupS6m6ftuzPq7")
project = rf.workspace("karol-stefany-saavedra-suarez").project("eggs-xa2gd-aqe53")
version = project.version(3)
dataset = version.download("yolov8")
                

loading Roboflow workspace...
loading Roboflow project...


In [7]:
print(dataset.location)

/home/ubuntu/huevos/eggs-3


In [8]:
import os

print(os.listdir(dataset.location))

['README.dataset.txt', 'train', 'data.yaml', 'valid', 'test', 'README.roboflow.txt']


In [6]:
from pathlib import Path

dataset_path = Path("/home/ubuntu/huevos/eggs-3")

print((dataset_path / "data.yaml").read_text())

names:
- Buenos
- Rotos
nc: 2
roboflow:
  license: CC BY 4.0
  project: eggs-xa2gd-aqe53
  url: https://universe.roboflow.com/karol-stefany-saavedra-suarez/eggs-xa2gd-aqe53/dataset/3
  version: 3
  workspace: karol-stefany-saavedra-suarez
test: ../test/images
train: ../train/images
val: ../valid/images



# Comprobación de la clases

In [11]:
from collections import Counter
from pathlib import Path

for split in ["train", "valid", "test"]:
    labels_path = dataset_path / split / "labels"

    counter = Counter()

    for file in labels_path.glob("*.txt"):
        with open(file, "r") as f:
            for line in f:
                line = line.strip()

                if line:
                    class_id = line.split()[0]
                    counter[class_id] += 1

    print(f"\n{split.upper()}")
    print(f"Buenos (0): {counter['0']}")
    print(f"Rotos  (1): {counter['1']}")


TRAIN
Buenos (0): 1266
Rotos  (1): 716

VALID
Buenos (0): 171
Rotos  (1): 38

TEST
Buenos (0): 77
Rotos  (1): 14


# Comenzar entrenamiento

YOLO11 ya que está haciendo transfer learning.


Partimos de un modelo YOLO previamente entrenado (YOLO11) y lo adaptamos a:
Buenos
Rotos

Ese archivo es un modelo YOLO11 Nano (n) que ya fue preentrenado por Ultralytics con un dataset grande de imágenes, principalmente COCO, que contiene muchas categorías de objetos.

El modelo ya aprendió cosas generales de las imágenes, como:

bordes
formas
texturas
patrones
características visuales de objetos

Pero no sabe específicamente qué es un huevo roto.

In [12]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data=str(dataset_path / "data.yaml"),
    epochs=50,
    imgsz=640, #tamaño de las imagenes 640x640
    batch=8, #el modelo procesa aproximadamente 8 imágenes por lote antes de actualizar sus pesos.
    device="cpu",
    workers=2, #cuántos procesos se utilizan para preparar/cargar los datos.
    project="/home/ubuntu/huevos/resultados", #guardar los resultados en esta carpeta
    name="huevos_baseline",
    patience=15 #evitar el entrenamiento inncesario, durante 15 épocas consecutivas no hay una mejora relevante en la validación, YOLO puede detener el entrenamiento antes de llegar a las 50 épocas.
)

Ultralytics 8.4.155 🚀 Python-3.12.3 torch-2.14.0+cu130 CPU (Intel Xeon CPU E5-2686 v4 @ 2.30GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ubuntu/huevos/eggs-3/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=huevos_baseline, nbs=64, 

verificar creación documentos

In [1]:
from pathlib import Path

results_path = Path("/home/ubuntu/huevos/resultados/huevos_baseline")

print("Archivos generados:")
for file in results_path.iterdir():
    print(file.name)

Archivos generados:
train_batch1.jpg
val_batch1_pred.jpg
train_batch0.jpg
args.yaml
weights
confusion_matrix.png
train_batch2082.jpg
BoxP_curve.png
results.csv
train_batch2.jpg
BoxR_curve.png
BoxF1_curve.png
train_batch2080.jpg
val_batch1_labels.jpg
BoxPR_curve.png
confusion_matrix_normalized.png
labels.jpg
train_batch2081.jpg
results.png
val_batch0_labels.jpg
val_batch0_pred.jpg


In [2]:
from pathlib import Path

weights_path = Path("/home/ubuntu/huevos/resultados/huevos_baseline/weights")

print(list(weights_path.iterdir()))

[PosixPath('/home/ubuntu/huevos/resultados/huevos_baseline/weights/last.pt'), PosixPath('/home/ubuntu/huevos/resultados/huevos_baseline/weights/best.pt')]


# evaluar el modelo sobre TEST

In [7]:
from ultralytics import YOLO

model = YOLO(
    "/home/ubuntu/huevos/resultados/huevos_baseline/weights/best.pt"
)

metrics = model.val(
    data=str(dataset_path / "data.yaml"),
    split="test",
    imgsz=640,
    batch=8,
    device="cpu",
    workers=2
)

Ultralytics 8.4.155 🚀 Python-3.12.3 torch-2.14.0+cu130 CPU (Intel Xeon CPU E5-2686 v4 @ 2.30GHz)
YOLO11n summary (fused): 100 layers, 2,582,542 parameters, 0 gradients, 6.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 24.1±18.7 MB/s, size: 20.3 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /home/ubuntu/huevos/eggs-3/test/labels.cache... 17 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 17/17 4.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.0s/it 3.1s2.9s
                   all         17         91      0.804      0.854      0.879      0.505
                Buenos          7         77      0.912      0.922      0.975      0.684
                 Rotos         10         14      0.696      0.786      0.783      0.326
Speed: 6.2ms preprocess, 156.3ms inference, 0.0ms l

Precision de Rotos = 69.6 %

De las detecciones que el modelo dice que son Rotos, aproximadamente 7 de cada 10 son correctas.

Recall de Rotos = 78.6 %

De los 14 huevos Rotos que realmente existen en TEST, el modelo consiguió detectar aproximadamente 11.

In [8]:
from ultralytics import YOLO

model = YOLO(
    "/home/ubuntu/huevos/resultados/huevos_baseline/weights/best.pt"
)

results = model.predict(
    source="/home/ubuntu/huevos/eggs-3/test/images",
    imgsz=640,
    conf=0.25,
    device="cpu",
    save=True,
    project="/home/ubuntu/huevos/predicciones",
    name="baseline_test"
)


image 1/17 /home/ubuntu/huevos/eggs-3/test/images/25_jpg.rf.4b43c4e27212334ea245c23145eadbd6.jpg: 608x640 1 Rotos, 161.3ms
image 2/17 /home/ubuntu/huevos/eggs-3/test/images/34_jpg.rf.29248def946230bc91854b856c8630e5.jpg: 448x640 1 Rotos, 117.0ms
image 3/17 /home/ubuntu/huevos/eggs-3/test/images/40_jpg.rf.7f06db066062d7100da84daabda49345.jpg: 384x640 3 Rotoss, 106.1ms
image 4/17 /home/ubuntu/huevos/eggs-3/test/images/43_jpg.rf.b84ecd0a10acfc6d4da94c8fa0ed5c47.jpg: 576x640 1 Rotos, 154.0ms
image 5/17 /home/ubuntu/huevos/eggs-3/test/images/5_PNG.rf.f6c7e56efd4db84968c9d8ef6ba1f493.jpg: 352x640 30 Buenoss, 97.9ms
image 6/17 /home/ubuntu/huevos/eggs-3/test/images/61_jpeg.rf.db6141d376ddbd392853ee82b8f9d76a.jpg: 640x480 3 Rotoss, 136.1ms
image 7/17 /home/ubuntu/huevos/eggs-3/test/images/63_jpeg.rf.ac519c3c0ce050ba7e00c66752431cf8.jpg: 480x640 1 Rotos, 130.7ms
image 8/17 /home/ubuntu/huevos/eggs-3/test/images/67_jpeg.rf.4898279c5f6b499d9b93e4dbb4e51483.jpg: 448x640 (no detections), 106.5ms
i

# Entrenamiento 2

In [1]:
#!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="G9twKyBupS6m6ftuzPq7")
project = rf.workspace("karol-stefany-saavedra-suarez").project("eggs-xa2gd-aqe53")
version = project.version(4)
dataset = version.download("yolov8")
                

loading Roboflow workspace...
loading Roboflow project...


In [4]:
from roboflow import Roboflow

rf = Roboflow(api_key="G9twKyBupS6m6ftuzPq7")

project = rf.workspace(
    "karol-stefany-saavedra-suarez"
).project(
    "eggs-xa2gd-aqe53"
)

version = project.version(4)

dataset_v4 = version.download("yolov8")

print("Dataset descargado en:")
print(dataset_v4.location)

loading Roboflow workspace...


loading Roboflow project...
Dataset descargado en:
/home/ubuntu/huevos/eggs-4


Definir la ruta

In [5]:
from pathlib import Path

dataset_v4_path = Path(dataset_v4.location)

print(dataset_v4_path)

/home/ubuntu/huevos/eggs-4


Comprobamos el nuevo dataset

In [7]:
print((dataset_v4_path / "data.yaml").read_text())

names:
- Buenos
- Rotos
nc: 2
roboflow:
  license: CC BY 4.0
  project: eggs-xa2gd-aqe53
  url: https://universe.roboflow.com/karol-stefany-saavedra-suarez/eggs-xa2gd-aqe53/dataset/4
  version: 4
  workspace: karol-stefany-saavedra-suarez
test: ../test/images
train: ../train/images
val: ../valid/images



Comparamos cuántos objetos hay ahora

In [8]:
from collections import Counter

for split in ["train", "valid", "test"]:
    labels_path = dataset_v4_path / split / "labels"

    counter = Counter()

    for file in labels_path.glob("*.txt"):
        with open(file, "r") as f:
            for line in f:
                line = line.strip()

                if line:
                    class_id = line.split()[0]
                    counter[class_id] += 1

    print(f"\n{split.upper()}")
    print(f"Buenos (0): {counter['0']}")
    print(f"Rotos  (1): {counter['1']}")


TRAIN
Buenos (0): 1010
Rotos  (1): 817

VALID
Buenos (0): 25
Rotos  (1): 38

TEST
Buenos (0): 10
Rotos  (1): 14


entrenar v2

In [9]:
from ultralytics import YOLO

model_v2 = YOLO("yolo11n.pt")

results_v2 = model_v2.train(
    data=str(dataset_v4_path / "data.yaml"),
    epochs=60,
    imgsz=640,
    batch=8,
    device="cpu",
    workers=2,
    project="/home/ubuntu/huevos/resultados",
    name="huevos_v2",
    patience=15
)

Ultralytics 8.4.155 🚀 Python-3.12.3 torch-2.14.0+cu130 CPU (Intel Xeon CPU E5-2686 v4 @ 2.30GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ubuntu/huevos/eggs-4/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=huevos_v2, nbs=64, nms=No

Evaluar entrenamiento

In [10]:
from ultralytics import YOLO

model_v2 = YOLO(
    "/home/ubuntu/huevos/resultados/huevos_v2/weights/best.pt"
)

metrics_v2 = model_v2.val(
    data=str(dataset_v4_path / "data.yaml"),
    split="test",
    imgsz=640,
    batch=8,
    device="cpu",
    workers=2
)

Ultralytics 8.4.155 🚀 Python-3.12.3 torch-2.14.0+cu130 CPU (Intel Xeon CPU E5-2686 v4 @ 2.30GHz)
YOLO11n summary (fused): 100 layers, 2,582,542 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 752.4±408.0 MB/s, size: 14.3 KB)
val: Scanning /home/ubuntu/huevos/eggs-4/test/labels... 14 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 14/14 3.4Kit/s 0.0s
val: New cache created: /home/ubuntu/huevos/eggs-4/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.5s/it 3.0s4.5s
                   all         14         24      0.626       0.65      0.665      0.377
                Buenos          4         10      0.561        0.8      0.647      0.446
                 Rotos         10         14      0.691        0.5      0.683      0.307
Speed: 7.0ms preprocess, 194.6ms inference, 0.0ms loss, 2.7ms postprocess per image
Results saved to /home/ubuntu/huevos/runs/detect/val-3


Entrenamiento 3

In [2]:
from pathlib import Path

dataset_v4_path = Path("/home/ubuntu/huevos/eggs-4")

print(dataset_v4_path)
print((dataset_v4_path / "data.yaml").exists())

/home/ubuntu/huevos/eggs-4
True


In [2]:
print((dataset_v4_path / "data.yaml").read_text())

names:
- Buenos
- Rotos
nc: 2
roboflow:
  license: CC BY 4.0
  project: eggs-xa2gd-aqe53
  url: https://universe.roboflow.com/karol-stefany-saavedra-suarez/eggs-xa2gd-aqe53/dataset/4
  version: 4
  workspace: karol-stefany-saavedra-suarez
test: ../test/images
train: ../train/images
val: ../valid/images



In [3]:
from collections import Counter

for split in ["train", "valid", "test"]:
    labels_path = dataset_v4_path / split / "labels"

    counter = Counter()

    for file in labels_path.glob("*.txt"):
        with open(file, "r") as f:
            for line in f:
                line = line.strip()

                if line:
                    class_id = line.split()[0]
                    counter[class_id] += 1

    print(f"\n{split.upper()}")
    print(f"Buenos (0): {counter['0']}")
    print(f"Rotos  (1): {counter['1']}")


TRAIN
Buenos (0): 1010
Rotos  (1): 817

VALID
Buenos (0): 25
Rotos  (1): 38

TEST
Buenos (0): 10
Rotos  (1): 14


In [5]:
from ultralytics import YOLO

model_v3 = YOLO("yolo11n.pt")

results_v3 = model_v3.train(
    data=str(dataset_v4_path / "data.yaml"),
    epochs=60,
    imgsz=640,
    batch=8,
    device="cpu",
    workers=2,
    project="/home/ubuntu/huevos/resultados",
    name="huevos_v3",
    patience=15
)

New https://pypi.org/project/ultralytics/8.4.156 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.155 🚀 Python-3.12.3 torch-2.14.0+cu130 CPU (Intel Xeon CPU E5-2686 v4 @ 2.30GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ubuntu/huevos/eggs-4/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mod

In [4]:
from ultralytics import YOLO

model_v3 = YOLO(
    "/home/ubuntu/huevos/resultados/huevos_v3/weights/best.pt"
)

metrics_v3 = model_v3.val(
    data=str(dataset_v4_path / "data.yaml"),
    split="test",
    imgsz=640,
    batch=8,
    device="cpu",
    workers=2
)

Ultralytics 8.4.155 🚀 Python-3.12.3 torch-2.14.0+cu130 CPU (Intel Xeon CPU E5-2686 v4 @ 2.30GHz)
YOLO11n summary (fused): 100 layers, 2,582,542 parameters, 0 gradients, 6.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 17.9±11.9 MB/s, size: 18.4 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /home/ubuntu/huevos/eggs-4/test/labels.cache... 14 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 14/14 5.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.1s/it 2.2s3.6s
                   all         14         24      0.626       0.65      0.665      0.377
                Buenos          4         10      0.561        0.8      0.647      0.446
                 Rotos         10         14      0.691        0.5      0.683      0.307
Speed: 2.0ms preprocess, 139.9ms inference, 0.0ms l

Revisar y comparar imágenes

c=0.10
detecta los marrones pero con algunoes errores de doble detección en una misma imagnes 

In [5]:
from ultralytics import YOLO

model_v3 = YOLO(
    "/home/ubuntu/huevos/resultados/huevos_v3/weights/best.pt"
)

results = model_v3.predict(
    source="/home/ubuntu/huevos/eggs-4/test/images",
    imgsz=640,
    conf=0.10,
    device="cpu",
    save=True,
    project="/home/ubuntu/huevos/predicciones",
    name="v3_test_conf010"
)


image 1/14 /home/ubuntu/huevos/eggs-4/test/images/25_jpg.rf.4b43c4e27212334ea245c23145eadbd6.jpg: 608x640 5 Rotoss, 272.0ms
image 2/14 /home/ubuntu/huevos/eggs-4/test/images/34_jpg.rf.29248def946230bc91854b856c8630e5.jpg: 448x640 10 Rotoss, 206.4ms
image 3/14 /home/ubuntu/huevos/eggs-4/test/images/40_jpg.rf.7f06db066062d7100da84daabda49345.jpg: 384x640 6 Rotoss, 114.4ms
image 4/14 /home/ubuntu/huevos/eggs-4/test/images/43_jpg.rf.b84ecd0a10acfc6d4da94c8fa0ed5c47.jpg: 576x640 3 Rotoss, 160.1ms
image 5/14 /home/ubuntu/huevos/eggs-4/test/images/61_jpeg.rf.db6141d376ddbd392853ee82b8f9d76a.jpg: 640x480 7 Rotoss, 131.6ms
image 6/14 /home/ubuntu/huevos/eggs-4/test/images/63_jpeg.rf.ac519c3c0ce050ba7e00c66752431cf8.jpg: 480x640 2 Rotoss, 123.2ms
image 7/14 /home/ubuntu/huevos/eggs-4/test/images/67_jpeg.rf.4898279c5f6b499d9b93e4dbb4e51483.jpg: 448x640 1 Buenos, 1 Rotos, 102.1ms
image 8/14 /home/ubuntu/huevos/eggs-4/test/images/84_jpeg.rf.2a02964d99dd1c8e0313fa59a1728347.jpg: 512x640 3 Buenoss, 

In [6]:
from ultralytics import YOLO

model_v3 = YOLO(
    "/home/ubuntu/huevos/resultados/huevos_v3/weights/best.pt"
)

results = model_v3.predict(
    source="/home/ubuntu/huevos/eggs-4/test/images",
    imgsz=640,
    conf=0.25,
    device="cpu",
    save=True,
    project="/home/ubuntu/huevos/predicciones",
    name="v3_test_conf025"
)


image 1/14 /home/ubuntu/huevos/eggs-4/test/images/25_jpg.rf.4b43c4e27212334ea245c23145eadbd6.jpg: 608x640 1 Rotos, 143.1ms
image 2/14 /home/ubuntu/huevos/eggs-4/test/images/34_jpg.rf.29248def946230bc91854b856c8630e5.jpg: 448x640 4 Rotoss, 108.5ms
image 3/14 /home/ubuntu/huevos/eggs-4/test/images/40_jpg.rf.7f06db066062d7100da84daabda49345.jpg: 384x640 3 Rotoss, 93.3ms
image 4/14 /home/ubuntu/huevos/eggs-4/test/images/43_jpg.rf.b84ecd0a10acfc6d4da94c8fa0ed5c47.jpg: 576x640 1 Rotos, 140.3ms
image 5/14 /home/ubuntu/huevos/eggs-4/test/images/61_jpeg.rf.db6141d376ddbd392853ee82b8f9d76a.jpg: 640x480 4 Rotoss, 112.6ms
image 6/14 /home/ubuntu/huevos/eggs-4/test/images/63_jpeg.rf.ac519c3c0ce050ba7e00c66752431cf8.jpg: 480x640 1 Rotos, 115.5ms
image 7/14 /home/ubuntu/huevos/eggs-4/test/images/67_jpeg.rf.4898279c5f6b499d9b93e4dbb4e51483.jpg: 448x640 (no detections), 107.0ms
image 8/14 /home/ubuntu/huevos/eggs-4/test/images/84_jpeg.rf.2a02964d99dd1c8e0313fa59a1728347.jpg: 512x640 1 Buenos, 121.2ms


In [7]:
results = model_v3.predict(
    source="/home/ubuntu/huevos/eggs-4/test/images",
    imgsz=640,
    conf=0.50,
    device="cpu",
    save=True,
    project="/home/ubuntu/huevos/predicciones",
    name="v3_test_conf050"
)


image 1/14 /home/ubuntu/huevos/eggs-4/test/images/25_jpg.rf.4b43c4e27212334ea245c23145eadbd6.jpg: 608x640 (no detections), 386.6ms
image 2/14 /home/ubuntu/huevos/eggs-4/test/images/34_jpg.rf.29248def946230bc91854b856c8630e5.jpg: 448x640 (no detections), 168.0ms
image 3/14 /home/ubuntu/huevos/eggs-4/test/images/40_jpg.rf.7f06db066062d7100da84daabda49345.jpg: 384x640 (no detections), 93.0ms
image 4/14 /home/ubuntu/huevos/eggs-4/test/images/43_jpg.rf.b84ecd0a10acfc6d4da94c8fa0ed5c47.jpg: 576x640 (no detections), 138.8ms
image 5/14 /home/ubuntu/huevos/eggs-4/test/images/61_jpeg.rf.db6141d376ddbd392853ee82b8f9d76a.jpg: 640x480 1 Rotos, 118.2ms
image 6/14 /home/ubuntu/huevos/eggs-4/test/images/63_jpeg.rf.ac519c3c0ce050ba7e00c66752431cf8.jpg: 480x640 (no detections), 111.6ms
image 7/14 /home/ubuntu/huevos/eggs-4/test/images/67_jpeg.rf.4898279c5f6b499d9b93e4dbb4e51483.jpg: 448x640 (no detections), 107.8ms
image 8/14 /home/ubuntu/huevos/eggs-4/test/images/84_jpeg.rf.2a02964d99dd1c8e0313fa59a17

In [8]:
results = model_v3.predict(
    source="/home/ubuntu/huevos/eggs-4/test/images",
    imgsz=640,
    conf=0.10,
    iou=0.50,
    device="cpu",
    save=True,
    project="/home/ubuntu/huevos/predicciones",
    name="v3_conf010_iou050"
)


image 1/14 /home/ubuntu/huevos/eggs-4/test/images/25_jpg.rf.4b43c4e27212334ea245c23145eadbd6.jpg: 608x640 3 Rotoss, 224.7ms
image 2/14 /home/ubuntu/huevos/eggs-4/test/images/34_jpg.rf.29248def946230bc91854b856c8630e5.jpg: 448x640 8 Rotoss, 112.7ms
image 3/14 /home/ubuntu/huevos/eggs-4/test/images/40_jpg.rf.7f06db066062d7100da84daabda49345.jpg: 384x640 4 Rotoss, 99.4ms
image 4/14 /home/ubuntu/huevos/eggs-4/test/images/43_jpg.rf.b84ecd0a10acfc6d4da94c8fa0ed5c47.jpg: 576x640 3 Rotoss, 131.5ms
image 5/14 /home/ubuntu/huevos/eggs-4/test/images/61_jpeg.rf.db6141d376ddbd392853ee82b8f9d76a.jpg: 640x480 5 Rotoss, 113.9ms
image 6/14 /home/ubuntu/huevos/eggs-4/test/images/63_jpeg.rf.ac519c3c0ce050ba7e00c66752431cf8.jpg: 480x640 1 Rotos, 115.4ms
image 7/14 /home/ubuntu/huevos/eggs-4/test/images/67_jpeg.rf.4898279c5f6b499d9b93e4dbb4e51483.jpg: 448x640 1 Buenos, 1 Rotos, 107.8ms
image 8/14 /home/ubuntu/huevos/eggs-4/test/images/84_jpeg.rf.2a02964d99dd1c8e0313fa59a1728347.jpg: 512x640 2 Buenoss, 1 R

In [9]:
from ultralytics import YOLO
from pathlib import Path
from collections import Counter

# Cargar nuestro mejor modelo actual
model_v3 = YOLO(
    "/home/ubuntu/huevos/resultados/huevos_v3/weights/best.pt"
)

# Carpeta de imágenes TEST de Roboflow V4
test_path = Path("/home/ubuntu/huevos/eggs-4/test/images")

# Ejecutar predicciones
results = model_v3.predict(
    source=str(test_path),
    imgsz=640,
    conf=0.10,
    iou=0.50,
    device="cpu",
    verbose=False
)

print(f"Imágenes analizadas: {len(results)}")

# Contadores
total_detections = 0
class_counter = Counter()
confidence_by_class = {
    "Buenos": [],
    "Rotos": []
}

for result in results:

    boxes = result.boxes

    if boxes is None:
        continue

    for cls, conf in zip(boxes.cls.tolist(), boxes.conf.tolist()):

        class_id = int(cls)
        confidence = float(conf)

        if class_id == 0:
            class_name = "Buenos"
        elif class_id == 1:
            class_name = "Rotos"
        else:
            class_name = f"Clase_{class_id}"

        total_detections += 1
        class_counter[class_name] += 1

        if class_name in confidence_by_class:
            confidence_by_class[class_name].append(confidence)

print("\n========== RESULTADOS ==========")
print(f"Total de detecciones: {total_detections}")

print("\nDetecciones por clase:")
for class_name, count in class_counter.items():
    print(f"{class_name}: {count}")

print("\nConfianza promedio:")

for class_name, values in confidence_by_class.items():

    if values:
        average = sum(values) / len(values)
        minimum = min(values)
        maximum = max(values)

        print(
            f"{class_name}: "
            f"promedio={average:.3f} | "
            f"mínima={minimum:.3f} | "
            f"máxima={maximum:.3f}"
        )

Imágenes analizadas: 14

========== RESULTADOS ==========
Total de detecciones: 73

Detecciones por clase:
Rotos: 38
Buenos: 35

Confianza promedio:
Buenos: promedio=0.336 | mínima=0.106 | máxima=0.720
Rotos: promedio=0.242 | mínima=0.102 | máxima=0.557


In [10]:
print("\n========== DETECCIONES DE BAJA CONFIANZA ==========")

for result in results:

    image_name = Path(result.path).name

    if result.boxes is None:
        continue

    for cls, conf in zip(
        result.boxes.cls.tolist(),
        result.boxes.conf.tolist()
    ):

        class_id = int(cls)
        confidence = float(conf)

        if class_id == 0:
            class_name = "Buenos"
        elif class_id == 1:
            class_name = "Rotos"
        else:
            class_name = f"Clase_{class_id}"

        if confidence < 0.20:

            print(
                f"{image_name} → "
                f"{class_name} → "
                f"confianza={confidence:.3f}"
            )


========== DETECCIONES DE BAJA CONFIANZA ==========
25_jpg.rf.4b43c4e27212334ea245c23145eadbd6.jpg → Rotos → confianza=0.117
34_jpg.rf.29248def946230bc91854b856c8630e5.jpg → Rotos → confianza=0.123
34_jpg.rf.29248def946230bc91854b856c8630e5.jpg → Rotos → confianza=0.106
34_jpg.rf.29248def946230bc91854b856c8630e5.jpg → Rotos → confianza=0.104
34_jpg.rf.29248def946230bc91854b856c8630e5.jpg → Rotos → confianza=0.102
40_jpg.rf.7f06db066062d7100da84daabda49345.jpg → Rotos → confianza=0.155
40_jpg.rf.7f06db066062d7100da84daabda49345.jpg → Rotos → confianza=0.109
43_jpg.rf.b84ecd0a10acfc6d4da94c8fa0ed5c47.jpg → Rotos → confianza=0.180
43_jpg.rf.b84ecd0a10acfc6d4da94c8fa0ed5c47.jpg → Rotos → confianza=0.141
61_jpeg.rf.db6141d376ddbd392853ee82b8f9d76a.jpg → Rotos → confianza=0.181
61_jpeg.rf.db6141d376ddbd392853ee82b8f9d76a.jpg → Rotos → confianza=0.153
67_jpeg.rf.4898279c5f6b499d9b93e4dbb4e51483.jpg → Buenos → confianza=0.110
84_jpeg.rf.2a02964d99dd1c8e0313fa59a1728347.jpg → Rotos → confianza

In [11]:
print("\n========== IMÁGENES CON MÁS DETECCIONES ==========")

image_detection_count = []

for result in results:

    image_name = Path(result.path).name

    if result.boxes is None:
        count = 0
    else:
        count = len(result.boxes)

    image_detection_count.append(
        (image_name, count)
    )

image_detection_count.sort(
    key=lambda x: x[1],
    reverse=True
)

for image_name, count in image_detection_count[:10]:

    print(
        f"{image_name}: "
        f"{count} detecciones"
    )


========== IMÁGENES CON MÁS DETECCIONES ==========
QXBp19_jpg.rf.83105faa04dfa69e814c4b6eaf902d99.jpg: 17 detecciones
RiZwaWQ9QXBp_jpg.rf.3b4693dfb3887d628e93632733ca8550.jpg: 12 detecciones
34_jpg.rf.29248def946230bc91854b856c8630e5.jpg: 8 detecciones
h4_jpg.rf.d79e6f3477ccd52726b5ef33354d0143.jpg: 8 detecciones
61_jpeg.rf.db6141d376ddbd392853ee82b8f9d76a.jpg: 5 detecciones
40_jpg.rf.7f06db066062d7100da84daabda49345.jpg: 4 detecciones
25_jpg.rf.4b43c4e27212334ea245c23145eadbd6.jpg: 3 detecciones
43_jpg.rf.b84ecd0a10acfc6d4da94c8fa0ed5c47.jpg: 3 detecciones
84_jpeg.rf.2a02964d99dd1c8e0313fa59a1728347.jpg: 3 detecciones
95_jpeg.rf.37ea87aacfede2b5463de5fecda35226.jpg: 3 detecciones


In [12]:
from pathlib import Path

print("\n========== DETECCIONES ==========")

for result in results:

    image_name = Path(result.path).name

    if result.boxes is None:
        continue

    print(f"\n📷 {image_name}")

    for i, (box, cls, conf) in enumerate(
        zip(
            result.boxes.xyxy.tolist(),
            result.boxes.cls.tolist(),
            result.boxes.conf.tolist()
        )
    ):

        class_id = int(cls)

        if class_id == 0:
            class_name = "Buenos"
        elif class_id == 1:
            class_name = "Rotos"
        else:
            class_name = f"Clase_{class_id}"

        x1, y1, x2, y2 = box

        print(
            f"  #{i+1} | "
            f"{class_name} | "
            f"conf={conf:.3f} | "
            f"box=({x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f})"
        )


========== DETECCIONES ==========

📷 25_jpg.rf.4b43c4e27212334ea245c23145eadbd6.jpg
  #1 | Rotos | conf=0.486 | box=(260, 162, 353, 311)
  #2 | Rotos | conf=0.226 | box=(318, 176, 352, 284)
  #3 | Rotos | conf=0.117 | box=(259, 174, 299, 266)

📷 34_jpg.rf.29248def946230bc91854b856c8630e5.jpg
  #1 | Rotos | conf=0.427 | box=(486, 227, 690, 308)
  #2 | Rotos | conf=0.267 | box=(134, 150, 668, 510)
  #3 | Rotos | conf=0.258 | box=(483, 259, 655, 302)
  #4 | Rotos | conf=0.214 | box=(114, 343, 403, 616)
  #5 | Rotos | conf=0.123 | box=(490, 379, 626, 438)
  #6 | Rotos | conf=0.106 | box=(232, 41, 686, 112)
  #7 | Rotos | conf=0.104 | box=(345, 168, 426, 343)
  #8 | Rotos | conf=0.102 | box=(487, 233, 696, 391)

📷 40_jpg.rf.7f06db066062d7100da84daabda49345.jpg
  #1 | Rotos | conf=0.463 | box=(1021, 266, 1249, 489)
  #2 | Rotos | conf=0.442 | box=(328, 217, 615, 475)
  #3 | Rotos | conf=0.155 | box=(371, 293, 614, 422)
  #4 | Rotos | conf=0.109 | box=(1071, 318, 1214, 447)

📷 43_jpg.rf.b84e

lógica de asociación Buenos ↔ Rotos en Python y probarla automáticamente sobre las 14 imágenes de TEST.

In [13]:
from ultralytics import YOLO
from pathlib import Path

model_v3 = YOLO(
    "/home/ubuntu/huevos/resultados/huevos_v3/weights/best.pt"
)

test_path = Path("/home/ubuntu/huevos/eggs-4/test/images")

results = model_v3.predict(
    source=str(test_path),
    imgsz=640,
    conf=0.10,
    iou=0.50,
    device="cpu",
    verbose=False
)

print(f"Imágenes analizadas: {len(results)}")

Imágenes analizadas: 14


función de asociación

In [14]:
def centro_caja(box):
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2, (y1 + y2) / 2)


def punto_dentro_caja(punto, caja):
    px, py = punto
    x1, y1, x2, y2 = caja

    return (
        x1 <= px <= x2
        and
        y1 <= py <= y2
    )

analizar imágenes

In [15]:
from pathlib import Path

resultados_huevos = []

for result in results:

    image_name = Path(result.path).name

    if result.boxes is None:
        resultados_huevos.append({
            "imagen": image_name,
            "huevos": []
        })
        continue

    boxes = result.boxes.xyxy.tolist()
    classes = result.boxes.cls.tolist()
    confidences = result.boxes.conf.tolist()

    huevos = []
    danos = []

    # Separar Buenos y Rotos
    for box, cls, conf in zip(boxes, classes, confidences):

        class_id = int(cls)

        if class_id == 0:
            huevos.append({
                "box": box,
                "confidence": float(conf)
            })

        elif class_id == 1:
            danos.append({
                "box": box,
                "confidence": float(conf)
            })

    # Asociar cada daño con un huevo
    huevos_detectados = []

    for i, huevo in enumerate(huevos):

        huevo_box = huevo["box"]

        danos_asociados = []

        for dano in danos:

            centro_dano = centro_caja(dano["box"])

            if punto_dentro_caja(centro_dano, huevo_box):

                danos_asociados.append(dano)

        # Si existe al menos un daño dentro del huevo
        if len(danos_asociados) > 0:
            estado = "ROTO"
        else:
            estado = "BUENO"

        huevos_detectados.append({
            "huevo": i + 1,
            "estado": estado,
            "confianza_huevo": huevo["confidence"],
            "danos": len(danos_asociados),
            "confianzas_danos": [
                d["confidence"]
                for d in danos_asociados
            ]
        })

    resultados_huevos.append({
        "imagen": image_name,
        "huevos": huevos_detectados
    })

In [16]:
print("\n========== CLASIFICACIÓN DE HUEVOS ==========\n")

for imagen in resultados_huevos:

    print(f"📷 {imagen['imagen']}")

    if not imagen["huevos"]:
        print("   No se detectaron huevos")
        continue

    for huevo in imagen["huevos"]:

        print(
            f"   🥚 Huevo {huevo['huevo']} → "
            f"{huevo['estado']} | "
            f"conf huevo={huevo['confianza_huevo']:.3f} | "
            f"zonas dañadas={huevo['danos']}"
        )

    print()


========== CLASIFICACIÓN DE HUEVOS ==========

📷 25_jpg.rf.4b43c4e27212334ea245c23145eadbd6.jpg
   No se detectaron huevos
📷 34_jpg.rf.29248def946230bc91854b856c8630e5.jpg
   No se detectaron huevos
📷 40_jpg.rf.7f06db066062d7100da84daabda49345.jpg
   No se detectaron huevos
📷 43_jpg.rf.b84ecd0a10acfc6d4da94c8fa0ed5c47.jpg
   No se detectaron huevos
📷 61_jpeg.rf.db6141d376ddbd392853ee82b8f9d76a.jpg
   No se detectaron huevos
📷 63_jpeg.rf.ac519c3c0ce050ba7e00c66752431cf8.jpg
   No se detectaron huevos
📷 67_jpeg.rf.4898279c5f6b499d9b93e4dbb4e51483.jpg
   🥚 Huevo 1 → BUENO | conf huevo=0.110 | zonas dañadas=0

📷 84_jpeg.rf.2a02964d99dd1c8e0313fa59a1728347.jpg
   🥚 Huevo 1 → ROTO | conf huevo=0.594 | zonas dañadas=1
   🥚 Huevo 2 → BUENO | conf huevo=0.211 | zonas dañadas=0

📷 95_jpeg.rf.37ea87aacfede2b5463de5fecda35226.jpg
   No se detectaron huevos
📷 QXBp19_jpg.rf.83105faa04dfa69e814c4b6eaf902d99.jpg
   🥚 Huevo 1 → BUENO | conf huevo=0.611 | zonas dañadas=0
   🥚 Huevo 2 → BUENO | conf hue

In [17]:
total_huevos = 0
total_buenos = 0
total_rotos = 0

for imagen in resultados_huevos:

    for huevo in imagen["huevos"]:

        total_huevos += 1

        if huevo["estado"] == "BUENO":
            total_buenos += 1

        elif huevo["estado"] == "ROTO":
            total_rotos += 1


print("========== RESUMEN ==========")
print(f"Total de huevos detectados: {total_huevos}")
print(f"Huevos BUENOS: {total_buenos}")
print(f"Huevos ROTOS: {total_rotos}")

========== RESUMEN ==========
Total de huevos detectados: 35
Huevos BUENOS: 34
Huevos ROTOS: 1


nueva versión roboflow (5)

In [1]:
#!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="G9twKyBupS6m6ftuzPq7")
project = rf.workspace("karol-stefany-saavedra-suarez").project("eggs-xa2gd-aqe53")
version = project.version(5)
dataset = version.download("yolov8")
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to eggs-5 in yolov8:: 100%|██████████| 991/991 [00:00<00:00, 5845.05it/s]


In [4]:
from pathlib import Path

dataset_path = Path(dataset.location)

print("Dataset:", dataset_path)
print(list(dataset_path.iterdir()))

NameError: name 'dataset' is not defined

In [3]:
from collections import Counter

for split in ["train", "valid", "test"]:

    labels_path = dataset_path / split / "labels"

    counter = Counter()

    for file in labels_path.glob("*.txt"):

        with open(file, "r") as f:

            for line in f:

                line = line.strip()

                if line:
                    class_id = line.split()[0]
                    counter[class_id] += 1

    print(f"\n{split.upper()}")
    print(f"Buenos (0): {counter['0']}")
    print(f"Rotos  (1): {counter['1']}")


TRAIN
Buenos (0): 932
Rotos  (1): 478

VALID
Buenos (0): 29
Rotos  (1): 26

TEST
Buenos (0): 11
Rotos  (1): 12


In [4]:
print((dataset_path / "data.yaml").read_text())

names:
- Buenos
- Rotos
nc: 2
roboflow:
  license: CC BY 4.0
  project: eggs-xa2gd-aqe53
  url: https://universe.roboflow.com/karol-stefany-saavedra-suarez/eggs-xa2gd-aqe53/dataset/5
  version: 5
  workspace: karol-stefany-saavedra-suarez
test: ../test/images
train: ../train/images
val: ../valid/images



# Nuevo entrenamiento

In [ ]:
from ultralytics import YOLO

model_v5 = YOLO("yolo11n.pt")

results_v5 = model_v5.train(

    data=str(dataset_path / "data.yaml"),
    epochs=70,
    imgsz=640,
    batch=8,
    device="cpu",
    workers=2,
    project="/home/ubuntu/huevos/resultados",
    name="huevos_v5",
    patience=20,
    pretrained=True,
    seed=42
)

New https://pypi.org/project/ultralytics/8.4.156 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.155 🚀 Python-3.12.3 torch-2.14.0+cu130 CPU (Intel Xeon CPU E5-2686 v4 @ 2.30GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ubuntu/huevos/eggs-5/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=70, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mod

In [2]:
from ultralytics import YOLO

model_v5 = YOLO(
    "/home/ubuntu/huevos/resultados/huevos_v5/weights/best.pt"
)

metrics_v5 = model_v5.val(
    data=str(dataset_path / "data.yaml"),
    split="test",
    imgsz=640,
    batch=8,
    device="cpu",
    workers=2
)

KeyboardInterrupt: 

In [ ]:
results_v5 = model_v5.predict(
    source=str(dataset_path / "test/images"),
    imgsz=640,
    conf=0.10,
    iou=0.50,
    device="cpu",
    verbose=False
)

Entrenamiento 6

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

PyTorch: 2.14.0+cu130
CUDA disponible: False


In [2]:
#!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="G9twKyBupS6m6ftuzPq7")
project = rf.workspace("karol-stefany-saavedra-suarez").project("eggs-xa2gd-aqe53")
version = project.version(6)
dataset = version.download("yolov8")
                

loading Roboflow workspace...
loading Roboflow project...


In [3]:
from pathlib import Path

dataset_path = Path(dataset.location)

print("Dataset:", dataset_path)
print(list(dataset_path.iterdir()))

Dataset: /home/ubuntu/huevos/eggs-6
[PosixPath('/home/ubuntu/huevos/eggs-6/README.dataset.txt'), PosixPath('/home/ubuntu/huevos/eggs-6/train'), PosixPath('/home/ubuntu/huevos/eggs-6/data.yaml'), PosixPath('/home/ubuntu/huevos/eggs-6/valid'), PosixPath('/home/ubuntu/huevos/eggs-6/test'), PosixPath('/home/ubuntu/huevos/eggs-6/README.roboflow.txt')]


In [4]:
print((dataset_path / "data.yaml").read_text())

train: ../train/images
val: ../valid/images
test: ../test/images

nc: 2
names: ['Buenos', 'Rotos']

roboflow:
  workspace: karol-stefany-saavedra-suarez
  project: eggs-xa2gd-aqe53
  version: 6
  license: CC BY 4.0
  url: https://universe.roboflow.com/karol-stefany-saavedra-suarez/eggs-xa2gd-aqe53/dataset/6


In [5]:
from pathlib import Path

print("========== REVISIÓN DE ETIQUETAS ==========")

problemas = []

for split in ["train", "valid", "test"]:

    labels_path = dataset_path / split / "labels"

    print(f"\nRevisando {split.upper()}...")

    for file in labels_path.glob("*.txt"):

        with open(file, "r") as f:
            lines = [line.strip() for line in f if line.strip()]

        for line_number, line in enumerate(lines, start=1):

            values = line.split()

            if len(values) != 5:

                problemas.append(
                    (split, file.name, line_number, len(values))
                )

                print(
                    f"⚠️ {file.name} | "
                    f"línea {line_number} | "
                    f"{len(values)} valores"
                )

print("\n========== RESULTADO ==========")

if not problemas:
    print("✅ Todas las etiquetas tienen formato Object Detection.")
else:
    print(f"❌ Se encontraron {len(problemas)} problemas.")

========== REVISIÓN DE ETIQUETAS ==========

Revisando TRAIN...
⚠️ 30_jpg.rf.5ca13e08b24b0c5ff4ed49fc66a4440e.txt | línea 1 | 55 valores
⚠️ 30_jpg.rf.3933f3e5d1a446e87437b4f4cd6c2350.txt | línea 1 | 55 valores
⚠️ 47_jpeg.rf.e127ece5ddc3c3ac47e1737a48203306.txt | línea 1 | 47 valores
⚠️ 47_jpeg.rf.f35161e06ee8687fae341fbc0fd360a3.txt | línea 1 | 57 valores
⚠️ 30_jpg.rf.16c7b02b189cc00050a5e7721dd1e6f6.txt | línea 1 | 53 valores
⚠️ 47_jpeg.rf.a7e99435e2a4dfba9d606af4f83afc61.txt | línea 1 | 59 valores

Revisando VALID...

Revisando TEST...
⚠️ 84_jpeg.rf.a6d87f6ffe03f169e4b4fb81a11f3520.txt | línea 1 | 63 valores
⚠️ 63_jpeg.rf.11daa758512e9dfc0bd4ed09e32f262f.txt | línea 1 | 45 valores

========== RESULTADO ==========
❌ Se encontraron 8 problemas.


In [6]:
from collections import Counter

for split in ["train", "valid", "test"]:

    labels_path = dataset_path / split / "labels"

    counter = Counter()

    for file in labels_path.glob("*.txt"):

        with open(file, "r") as f:

            for line in f:

                line = line.strip()

                if line:

                    class_id = line.split()[0]

                    counter[class_id] += 1

    print(f"\n{split.upper()}")
    print(f"Buenos (0): {counter['0']}")
    print(f"Rotos  (1): {counter['1']}")


TRAIN
Buenos (0): 981
Rotos  (1): 444

VALID
Buenos (0): 29
Rotos  (1): 25

TEST
Buenos (0): 11
Rotos  (1): 12


In [ ]:
from ultralytics import YOLO

model_v6 = YOLO("yolo11n.pt")

results_v6 = model_v6.train(

    data=str(dataset_path / "data.yaml"),
    epochs=70,
    imgsz=640,
    batch=8,
    device="cpu",
    workers=2,
    project="/home/ubuntu/huevos/resultados",
    name="huevos_v6",
    patience=20,
    pretrained=True,
    seed=42
)

New https://pypi.org/project/ultralytics/8.4.157 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.155 🚀 Python-3.12.3 torch-2.14.0+cu130 CPU (Intel Xeon CPU E5-2686 v4 @ 2.30GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ubuntu/huevos/eggs-6/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=70, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mod

  7                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  8                  -1  1    346112  ultralytics.nn.modules.block.C3k2            [256, 256, 1, True]           
  9                  -1  1    164608  ultralytics.nn.modules.block.SPPF            [256, 256, 5]                 
 10                  -1  1    249728  ultralytics.nn.modules.block.C2PSA           [256, 256, 1]                 
 11                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 12             [-1, 6]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 13                  -1  1    111296  ultralytics.nn.modules.block.C3k2            [384, 128, 1, False]          
 14                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 15             [-1, 4]  1         0  ultralytics.nn.modules.conv.Concat           [1]  

In [ ]:
from ultralytics import YOLO

model_v6 = YOLO(
    "/home/ubuntu/huevos/resultados/huevos_v6/weights/best.pt"
)

print("✅ Modelo V6 cargado")

FileNotFoundError: [Errno 2] No such file or directory: '/home/ubuntu/huevos/resultados/huevos_v6/weights/best.pt'

In [ ]:
metrics_test = model_v6.val(
    data=str(dataset_path / "data.yaml"),
    split="test",
    imgsz=640,
    batch=8,
    device="cpu",
    workers=2
)

In [ ]:
from pathlib import Path

dataset_path = Path("/home/ubuntu/huevos/eggs-6")

print(dataset_path)
print(dataset_path.exists())

Nuevo Dataset

In [1]:
#!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="G9twKyBupS6m6ftuzPq7")
project = rf.workspace("karol-stefany-saavedra-suarez").project("dthuevos")
version = project.version(1)
dataset = version.download("yolov8")
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Dthuevos-1 in yolov8:: 100%|██████████| 465/465 [00:00<00:00, 10514.53it/s]


In [2]:
print(dataset.location)

/home/ubuntu/huevos/Dthuevos-1


In [3]:
from pathlib import Path

dataset_path = Path(dataset.location)

for split in ["train", "valid", "test"]:
    labels_path = dataset_path / split / "labels"

    problemas = []

    for archivo in labels_path.glob("*.txt"):
        with open(archivo, "r") as f:
            for numero, linea in enumerate(f, 1):

                valores = linea.strip().split()

                if not valores:
                    continue

                if len(valores) != 5:
                    problemas.append(
                        f"{archivo.name} | línea {numero} | {len(valores)} valores"
                    )

    print(f"\n========== {split.upper()} ==========")

    if problemas:
        print(f"⚠️ Problemas encontrados: {len(problemas)}")
        for problema in problemas[:20]:
            print(problema)
    else:
        print("✅ Todas las etiquetas tienen formato YOLO de detección")


========== TRAIN ==========
⚠️ Problemas encontrados: 3
malomarron5_webp.rf.fe3349d343ec1c2d84646e7a1b712e28.txt | línea 1 | 63 valores
malomarron5_webp.rf.036f6bb1bd1b0b29e89bc24e6d14dbac.txt | línea 1 | 61 valores
malomarron5_webp.rf.f14877ac5eb8759984a2fde490755c6e.txt | línea 1 | 63 valores

========== VALID ==========
✅ Todas las etiquetas tienen formato YOLO de detección

========== TEST ==========
✅ Todas las etiquetas tienen formato YOLO de detección


In [4]:
import yaml

with open(dataset_path / "data.yaml", "r") as f:
    data = yaml.safe_load(f)

print(data)

{'names': ['Bueno', 'Roto'], 'nc': 2, 'roboflow': {'license': 'CC BY 4.0', 'project': 'dthuevos', 'url': 'https://universe.roboflow.com/karol-stefany-saavedra-suarez/dthuevos/dataset/1', 'version': 1, 'workspace': 'karol-stefany-saavedra-suarez'}, 'test': '../test/images', 'train': '../train/images', 'val': '../valid/images'}


In [6]:
from pathlib import Path

base = Path("/home/ubuntu/huevos")

for p in base.iterdir():
    if p.is_dir():
        print(p)

/home/ubuntu/huevos/runs
/home/ubuntu/huevos/dataset
/home/ubuntu/huevos/resultados
/home/ubuntu/huevos/__pycache__
/home/ubuntu/huevos/eggs-3
/home/ubuntu/huevos/eggs-4
/home/ubuntu/huevos/eggs-6
/home/ubuntu/huevos/predicciones
/home/ubuntu/huevos/eggs-5
/home/ubuntu/huevos/venv
/home/ubuntu/huevos/eggs-2
/home/ubuntu/huevos/api
/home/ubuntu/huevos/model
/home/ubuntu/huevos/Dthuevos-1


In [7]:
!find /home/ubuntu/huevos -maxdepth 4 -type d | grep -E "Dthuevos|dthuevos|train|valid|test"

/home/ubuntu/huevos/eggs-3/train
/home/ubuntu/huevos/eggs-3/train/images
/home/ubuntu/huevos/eggs-3/train/labels
/home/ubuntu/huevos/eggs-3/valid
/home/ubuntu/huevos/eggs-3/valid/images
/home/ubuntu/huevos/eggs-3/valid/labels
/home/ubuntu/huevos/eggs-3/test
/home/ubuntu/huevos/eggs-3/test/images
/home/ubuntu/huevos/eggs-3/test/labels
/home/ubuntu/huevos/eggs-4/train
/home/ubuntu/huevos/eggs-4/train/images
/home/ubuntu/huevos/eggs-4/train/labels
/home/ubuntu/huevos/eggs-4/valid
/home/ubuntu/huevos/eggs-4/valid/images
/home/ubuntu/huevos/eggs-4/valid/labels
/home/ubuntu/huevos/eggs-4/test
/home/ubuntu/huevos/eggs-4/test/images
/home/ubuntu/huevos/eggs-4/test/labels
/home/ubuntu/huevos/eggs-6/train
/home/ubuntu/huevos/eggs-6/train/images
/home/ubuntu/huevos/eggs-6/train/labels
/home/ubuntu/huevos/eggs-6/valid
/home/ubuntu/huevos/eggs-6/valid/images
/home/ubuntu/huevos/eggs-6/valid/labels
/home/ubuntu/huevos/eggs-6/test
/home/ubuntu/huevos/eggs-6/test/images
/home/ubuntu/huevos/eggs-6/test

In [8]:
print(dataset.location)

/home/ubuntu/huevos/Dthuevos-1


In [12]:
from pathlib import Path



In [14]:
dataset_path = Path("/home/ubuntu/huevos/Dthuevos-1")



In [15]:


print("TRAIN:", list((dataset_path / "train").iterdir()))
print("VALID:", (dataset_path / "valid").exists())
print("TEST:", (dataset_path / "test").exists())

TRAIN: [PosixPath('/home/ubuntu/huevos/Dthuevos-1/train/images'), PosixPath('/home/ubuntu/huevos/Dthuevos-1/train/labels')]
VALID: False
TEST: False


In [17]:
print(
    "Imágenes TRAIN:",
    len(list((dataset_path / "train" / "images").glob("*")))
)

print(
    "Etiquetas TRAIN:",
    len(list((dataset_path / "train" / "labels").glob("*.txt")))
)

Imágenes TRAIN: 231
Etiquetas TRAIN: 231


In [18]:
from roboflow import Roboflow

rf = Roboflow(api_key="G9twKyBupS6m6ftuzPq7")
project = rf.workspace("karol-stefany-saavedra-suarez").project("dthuevos")
version = project.version(1)

dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...


In [20]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="G9twKyBupS6m6ftuzPq7")
project = rf.workspace("karol-stefany-saavedra-suarez").project("dthuevos")
version = project.version(1)
dataset = version.download("yolov8")
                

loading Roboflow workspace...
loading Roboflow project...


In [21]:
from pathlib import Path
import shutil
import random

dataset_path = Path("/home/ubuntu/huevos/Dthuevos-1")

# Carpetas originales
images_src = dataset_path / "train" / "images"
labels_src = dataset_path / "train" / "labels"

# Crear nuevas carpetas
for split in ["train", "valid", "test"]:
    (dataset_path / split / "images").mkdir(parents=True, exist_ok=True)
    (dataset_path / split / "labels").mkdir(parents=True, exist_ok=True)

# Obtener imágenes
images = list(images_src.glob("*"))
random.seed(42)
random.shuffle(images)

# División 70/20/10
total = len(images)
train_end = int(total * 0.70)
valid_end = train_end + int(total * 0.20)

train_images = images[:train_end]
valid_images = images[train_end:valid_end]
test_images = images[valid_end:]

print("Total:", total)
print("Train:", len(train_images))
print("Valid:", len(valid_images))
print("Test:", len(test_images))

# Copiar imágenes y etiquetas
def copiar_archivos(lista, split):
    for img in lista:
        shutil.copy2(img, dataset_path / split / "images" / img.name)

        label = labels_src / f"{img.stem}.txt"

        if label.exists():
            shutil.copy2(
                label,
                dataset_path / split / "labels" / label.name
            )

copiar_archivos(train_images, "train")
copiar_archivos(valid_images, "valid")
copiar_archivos(test_images, "test")

print("\n✅ Dataset dividido correctamente.")

Total: 231
Train: 161
Valid: 46
Test: 24


SameFileError: PosixPath('/home/ubuntu/huevos/Dthuevos-1/train/images/mr7_webp.rf.65fa471ddb4030f62e71e87c59a9ef02.jpg') and PosixPath('/home/ubuntu/huevos/Dthuevos-1/train/images/mr7_webp.rf.65fa471ddb4030f62e71e87c59a9ef02.jpg') are the same file

In [22]:
import shutil

dataset_path = Path("/home/ubuntu/huevos/Dthuevos-1")

shutil.rmtree(dataset_path / "valid", ignore_errors=True)
shutil.rmtree(dataset_path / "test", ignore_errors=True)

print("✅ Carpetas valid y test limpiadas")

✅ Carpetas valid y test limpiadas


In [23]:
from pathlib import Path
import shutil
import random

dataset_path = Path("/home/ubuntu/huevos/Dthuevos-1")

images_src = dataset_path / "train" / "images"
labels_src = dataset_path / "train" / "labels"

# Obtener las imágenes originales
images = list(images_src.glob("*"))

random.seed(42)
random.shuffle(images)

total = len(images)

train_end = int(total * 0.70)
valid_end = train_end + int(total * 0.20)

train_images = images[:train_end]
valid_images = images[train_end:valid_end]
test_images = images[valid_end:]

print("Total:", total)
print("Train:", len(train_images))
print("Valid:", len(valid_images))
print("Test:", len(test_images))


# Crear carpetas
for split in ["valid", "test"]:
    (dataset_path / split / "images").mkdir(parents=True, exist_ok=True)
    (dataset_path / split / "labels").mkdir(parents=True, exist_ok=True)


# Copiar valid y test
def copiar_archivos(lista, split):

    for img in lista:

        destino_img = dataset_path / split / "images" / img.name
        shutil.copy2(img, destino_img)

        label = labels_src / f"{img.stem}.txt"

        if label.exists():

            destino_label = dataset_path / split / "labels" / label.name
            shutil.copy2(label, destino_label)


copiar_archivos(valid_images, "valid")
copiar_archivos(test_images, "test")

print("\n✅ División terminada")

Total: 231
Train: 161
Valid: 46
Test: 24

✅ División terminada


In [24]:
for img in valid_images + test_images:

    img_train = images_src / img.name
    label_train = labels_src / f"{img.stem}.txt"

    if img_train.exists():
        img_train.unlink()

    if label_train.exists():
        label_train.unlink()

print("✅ Imágenes y etiquetas movidas correctamente")

✅ Imágenes y etiquetas movidas correctamente


In [25]:
for split in ["train", "valid", "test"]:

    imgs = list((dataset_path / split / "images").glob("*"))
    labels = list((dataset_path / split / "labels").glob("*.txt"))

    print(
        f"{split.upper()}: "
        f"{len(imgs)} imágenes | "
        f"{len(labels)} etiquetas"
    )

TRAIN: 161 imágenes | 161 etiquetas
VALID: 46 imágenes | 46 etiquetas
TEST: 24 imágenes | 24 etiquetas


In [26]:
from ultralytics import YOLO

model_v7 = YOLO("yolo11n.pt")

results_v7 = model_v7.train(
    data="/home/ubuntu/huevos/Dthuevos-1/data.yaml",
    epochs=70,
    imgsz=640,
    batch=8,
    patience=20,
    project="/home/ubuntu/huevos/resultados",
    name="huevos_v7",
    device="cpu"
)

New https://pypi.org/project/ultralytics/8.4.157 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.155 🚀 Python-3.12.3 torch-2.14.0+cu130 CPU (Intel Xeon CPU E5-2686 v4 @ 2.30GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/ubuntu/huevos/Dthuevos-1/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=70, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0,